# Model Evaluation

## Objective

In this notebook we evaluate the trained SRCNN model.

The evaluation is performed using the Peak Signal-to-Noise Ratio (PSNR), one of the most common metrics for image super-resolution.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

sys.path.append(str(project_root))

In [2]:
import torch

from src.models.srcnn import SRCNN
from src.dataloader import get_train_loader
from src.checkpoint import load_model
from src.metrics import calculate_psnr

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [4]:
train_loader = get_train_loader()

In [ ]:
model = SRCNN()

model = load_model(
    model,
    "../results/checkpoints/srcnn_x2_200epochs.pth",
    device
)

model.eval()

c:\Users\aless\Documents\RetroSR-CNN\src\checkpoint.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(filepath, map_location=device)


SRCNN(
  (feature_extraction): Sequential(
    (0): Conv2d(3, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
    (1): ReLU(inplace=True)
  )
  (mapping): Sequential(
    (0): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU(inplace=True)
  )
  (reconstruction): Conv2d(32, 3, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
)

In [6]:
lr_batch, hr_batch = next(iter(train_loader))

print(lr_batch.shape)

torch.Size([16, 3, 96, 96])


In [7]:
lr_batch = lr_batch.to(device)

with torch.no_grad():

    prediction = model(lr_batch)

In [8]:
bicubic_psnr = calculate_psnr(
    lr_batch,
    hr_batch.to(device)
)

print(bicubic_psnr)

28.21533203125


In [9]:
srcnn_psnr = calculate_psnr(
    prediction,
    hr_batch.to(device)
)

print(srcnn_psnr)

24.026538848876953


In [10]:
print("Prediction")

print(prediction.min().item())
print(prediction.max().item())

print()

print("Ground Truth")

print(hr_batch.min().item())
print(hr_batch.max().item())

Prediction
-0.03162273019552231
1.1881877183914185

Ground Truth
0.0
1.0


In [11]:
mse_prediction = torch.mean((prediction - hr_batch.to(device)) ** 2)

mse_bicubic = torch.mean((lr_batch - hr_batch.to(device)) ** 2)

print("Prediction MSE:", mse_prediction.item())
print("Bicubic MSE:", mse_bicubic.item())

Prediction MSE: 0.003956817556172609
Bicubic MSE: 0.0015082276659086347


In [12]:
print(lr_batch.min().item(), lr_batch.max().item())
print(hr_batch.min().item(), hr_batch.max().item())

0.0 1.0
0.0 1.0


In [13]:
difference = torch.mean(torch.abs(prediction - lr_batch))

print(difference.item())

0.026201393455266953


In [14]:
difference = torch.mean(torch.abs(prediction - hr_batch.to(device)))

print(difference.item())

0.038133732974529266
